# Test multilingual training pipeline
Notebook này kiểm tra từng phần mà không chạy toàn bộ training. Sau mỗi bước process data, một mẫu đã chuẩn hóa sẽ được in ra để kiểm tra.

In [ ]:
from pathlib import Path
import json
from itertools import islice

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent

MODEL_NAME = 'Qwen/Qwen2.5-0.5B'
LANGUAGE_PAIR = 'vi-en'
DIRECTION = 'both'
INSTRUCTION_LANGUAGE = 'vi'
MAX_LENGTH = 256
ALIGNMENT_MAX_LENGTH = 64
print('Repository:', REPO_ROOT)

## 1. Process Stage 1 data và xem mẫu đã xử lý

In [ ]:
from datasets import Dataset
from src.prepare_data import _discover_mt, _parallel_rows

train_paths = _discover_mt(REPO_ROOT / 'data' / 'MT', 'train', LANGUAGE_PAIR)
processed_rows = list(islice(_parallel_rows(train_paths, DIRECTION), 4))
stage1_sample_dataset = Dataset.from_list(processed_rows)

print('Files:', [str(path) for path in train_paths])
print('Number of sampled processed rows:', len(stage1_sample_dataset))
print('\nMẪU SAU KHI PROCESS_DATA:')
print(json.dumps(stage1_sample_dataset[0], ensure_ascii=False, indent=2))

## 2. Kiểm tra plain-text translation prompt

In [ ]:
from src.prompts import TRANSLATION_TARGET_MARKER, translation_instruction

sample = stage1_sample_dataset[0]
translation_prompt = (
    translation_instruction(sample['source_lang'], sample['target_lang'])
    + sample['source']
    + TRANSLATION_TARGET_MARKER
    + sample['target']
)
print(translation_prompt)

## 3. Tokenize, collate và kiểm tra lại source/target spans

In [ ]:
from transformers import AutoTokenizer
from src.collator import MultilingualDataCollator

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

alignment_collator = MultilingualDataCollator(
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
    alignment_max_length=ALIGNMENT_MAX_LENGTH,
)
batch = alignment_collator(processed_rows[:2])
print({key: tuple(value.shape) for key, value in batch.items()})

In [ ]:
row_index = 0
src_start = batch['source_start_positions'][row_index].item()
src_end = batch['source_end_positions'][row_index].item()
tgt_start = batch['target_start_positions'][row_index].item()
tgt_end = batch['target_end_positions'][row_index].item()
ids = batch['input_ids'][row_index]

print('Full sequence:')
print(tokenizer.decode(ids, skip_special_tokens=False))
print('\nSource span:', (src_start, src_end))
print(tokenizer.decode(ids[src_start:src_end], skip_special_tokens=False))
print('\nTarget span:', (tgt_start, tgt_end))
print(tokenizer.decode(ids[tgt_start:tgt_end], skip_special_tokens=False))
supervised_ids = batch['labels'][row_index][batch['labels'][row_index] != -100]
print('\nTokens được tính NTP loss:')
print(tokenizer.decode(supervised_ids, skip_special_tokens=False))

## 4. Chạy thử model Stage 1
Cell này tải model và chạy một forward pass. Có thể bỏ qua nếu chỉ muốn kiểm tra data.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from src.model import MultilingualAlignmentModel

model = MultilingualAlignmentModel(
    MODEL_NAME,
    contrastive_weight=0.1,
    ot_weight=0.05,
    align_layer=-1,
    attention_mass_weight=0.5,
    attn_implementation='eager',
)
model.lm = get_peft_model(model.lm, LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
))
model.lm.print_trainable_parameters()
output = model(**batch)
print('Total Stage 1 loss:', float(output.loss.detach()))
print('Logits shape:', tuple(output.logits.shape))

## 5. Mini train thử LoRA
Chạy ba optimizer steps trên sample nhỏ để kiểm tra backward, loss components và logging.

In [ ]:
import importlib
from transformers import TrainingArguments
import src.train as train_module
importlib.reload(train_module)
ComponentLoggingTrainer = train_module.ComponentLoggingTrainer

del output  # release the previous forward graph before training
mini_training_args = TrainingArguments(
    output_dir=str(REPO_ROOT / 'outputs' / 'notebook-smoke-test'),
    max_steps=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    logging_strategy='steps',
    logging_steps=1,
    logging_first_step=True,
    save_strategy='no',
    eval_strategy='no',
    report_to='none',
    remove_unused_columns=False,
)
mini_trainer = ComponentLoggingTrainer(
    model=model,
    args=mini_training_args,
    stage='alignment',
    train_dataset=stage1_sample_dataset,
    data_collator=alignment_collator,
)
mini_result = mini_trainer.train()
print('Mini-train metrics:', mini_result.metrics)

import pandas as pd
component_history = [
    row for row in mini_trainer.state.log_history
    if any(key.startswith('train/') for key in row)
]
display(pd.DataFrame(component_history))

## 6. Process Stage 2 data và xem mẫu XLSum/Bactrian

In [ ]:
import ijson
from src.prompts import summarization_instruction

xlsum_path = REPO_ROOT / 'data' / 'XLSum' / 'XLSum' / INSTRUCTION_LANGUAGE / f'validation.{INSTRUCTION_LANGUAGE}.json'
with xlsum_path.open('r', encoding='utf-8') as handle:
    raw_xlsum = json.loads(next(line for line in handle if line.strip()))
processed_xlsum = {
    'instruction': summarization_instruction(INSTRUCTION_LANGUAGE),
    'input': raw_xlsum['text'].strip(),
    'output': raw_xlsum['summary'].strip(),
    'language': INSTRUCTION_LANGUAGE,
    'dataset_name': 'xlsum',
}
print('MẪU XLSUM SAU KHI PROCESS_DATA:')
print(json.dumps(processed_xlsum, ensure_ascii=False, indent=2))

In [ ]:
bactrian_path = REPO_ROOT / 'data' / 'Bactrian-Multilingual_Instruction' / f'{INSTRUCTION_LANGUAGE}.json'
with bactrian_path.open('rb') as handle:
    raw_bactrian = next(ijson.items(handle, 'item'))
processed_bactrian = {
    'instruction': str(raw_bactrian['instruction']).strip(),
    'input': str(raw_bactrian.get('input') or '').strip(),
    'output': str(raw_bactrian['output']).strip(),
    'language': INSTRUCTION_LANGUAGE,
    'dataset_name': 'bactrian',
}
print('MẪU BACTRIAN SAU KHI PROCESS_DATA:')
print(json.dumps(processed_bactrian, ensure_ascii=False, indent=2))

## 7. Kiểm tra Stage 2 prompt và response-only labels

In [ ]:
from src.collator import InstructionDataCollator

instruction_collator = InstructionDataCollator(tokenizer, max_length=MAX_LENGTH)
instruction_batch = instruction_collator([processed_xlsum, processed_bactrian])
row_index = 0
ids = instruction_batch['input_ids'][row_index]
labels = instruction_batch['labels'][row_index]
print('Full Stage 2 sequence:')
print(tokenizer.decode(ids, skip_special_tokens=False))
print('\nChỉ phần sau được tính NTP loss:')
print(tokenizer.decode(labels[labels != -100], skip_special_tokens=False))

## 8. Load full datasets (tùy chọn)
Chỉ chạy cell dưới khi muốn tạo toàn bộ Hugging Face Dataset cache.

In [ ]:
# from src.prepare_data import load_parallel_dataset, load_instruction_dataset
# full_stage1 = load_parallel_dataset(
#     data_dir=str(REPO_ROOT / 'data' / 'MT'),
#     language_pairs=LANGUAGE_PAIR,
#     direction=DIRECTION,
# )
# print(full_stage1)
# print(full_stage1['train'][0])
# full_stage2 = load_instruction_dataset(languages=INSTRUCTION_LANGUAGE)
# print(full_stage2)
# print(full_stage2['train'][0])